In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
arquivo = 'Tabela4.csv'

idh = pd.read_csv(
    arquivo,
    sep=';',
    decimal=',',
    encoding='latin1',
    header=1
)
idh

In [ ]:
idh = idh.loc[:, idh.notna().any()]
idh.shape

In [ ]:
ranking_2024 = (
    idh.nlargest(len(idh), '2024')[['Sigla', 'Estado', '2024']]
    .reset_index(drop=True)
)
ranking_2024.index += 1
ranking_2024.index.name = 'Posição'
ranking_2024

In [ ]:

idh['ganho'] = (idh['2024'] - idh['1991']).round(3)

maiores_ganhos = idh.sort_values('ganho', ascending=False)
maiores_ganhos[['Sigla', 'Estado', '1991', '2024', 'ganho']]

In [ ]:
# algum estado terminou 2024 com IDH menor que o de 1991?
queda = idh[idh['ganho'] < 0]

if queda.empty:
    print('Nenhum estado piorou.')
else:
    display(queda[['Sigla', 'Estado', '1991', '2024', 'ganho']])

In [ ]:
# transforma a tabela para o formato longo (uma linha por estado/ano)
colunas_ano = idh.filter(regex=r'^\d{4}$').columns.tolist()
colunas_id = ['Sigla', 'Código', 'Estado']
print(colunas_ano)

idh_longo = pd.melt(
    idh,
    id_vars=colunas_id,
    value_vars=colunas_ano,
    var_name='Ano',
    value_name='IDH',
)
idh_longo['Ano'] = idh_longo['Ano'].astype(int)
idh_longo['IDH'] = pd.to_numeric(idh_longo['IDH'], errors='coerce')
idh_longo.head()

In [ ]:
# evolução do IDH em Minas Gerais
mg = idh_longo.query("Sigla == 'MG'").sort_values('Ano')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mg['Ano'], mg['IDH'], color='tab:green', marker='s', linewidth=2)

# marca o primeiro e o último valor da série
for _, linha in mg.iloc[[0, -1]].iterrows():
    ax.annotate(
        f"{linha['IDH']:.3f}",
        xy=(linha['Ano'], linha['IDH']),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
    )

ax.set_title('Evolução do IDH em Minas Gerais (1991–2024)')
ax.set_xlabel('Ano')
ax.set_ylabel('IDH')
ax.grid(linestyle='--', alpha=0.4)
plt.show()

In [ ]:
# evolução do IDH de todos os estados, com Minas Gerais em destaque
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in idh_longo.groupby('Sigla'):
    grupo = grupo.sort_values('Ano')
    eh_mg = sigla == 'MG'
    ax.plot(
        grupo['Ano'], grupo['IDH'],
        marker='o', markersize=3,
        linewidth=3 if eh_mg else 1.2,
        alpha=1.0 if eh_mg else 0.6,
        color='crimson' if eh_mg else None,
        zorder=3 if eh_mg else 2,
        label=sigla,
    )

ax.set_title('Evolução do IDH por estado (1991–2024)')
ax.set_xlabel('Ano')
ax.set_ylabel('IDH')
ax.set_ylim(0.3, 0.9)
ax.grid(linestyle='--', alpha=0.3)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc='upper left', title='UF')
fig.tight_layout()
plt.show()